<a href="https://colab.research.google.com/github/aniray2908/nlp-llm-journey/blob/main/00_pytorch_warmup/demos/02_autograd.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise Set 1 — Basic Gradients

## 1. Your first gradient

In [3]:
import torch
import torch.nn as nn

In [4]:
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2 + 5 * x + 2

In [5]:
y.backward()

In [6]:
x.grad

tensor(11.)

## 2. Slightly deeper

In [7]:
x = torch.tensor(4.0, requires_grad=True)
a = x ** 3
b = 2 * a
c = b - 10

In [8]:
c.backward()

In [9]:
x.grad

tensor(96.)

# Exercise Set 2 — The Computation Graph

## 3. Inspect the graph

In [10]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2
z = y * 4 + 1

In [11]:
z.grad_fn

In [12]:
z.grad_fn.next_functions

((<MulBackward0 at 0x7dfe98a786a0>, 0), (None, 0))

## 4. requires_grad behaviour

In [13]:
a = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(3.0)   # no grad tracking

c = a * b
d = c + a

In [14]:
c.grad_fn

In [15]:
b.grad_fn

In [16]:
d.backward()

In [17]:
a.grad

tensor(4.)

# Exercise Set 3 — Common Gotchas

## 5. The accumulation trap

In [27]:
x = torch.tensor(2.0, requires_grad=True)

for i in range(4):
    y = x ** 2
    y.backward()
    print(f"Step {i}: x.grad = {x.grad}") ##help
    x.grad.zero_()

Step 0: x.grad = 4.0
Step 1: x.grad = 4.0
Step 2: x.grad = 4.0
Step 3: x.grad = 4.0


## 6. no_grad in action

In [19]:
x = torch.tensor(2.0, requires_grad=True)

y = x ** 2
print(y.requires_grad)   # what is this?

with torch.no_grad():
    z = x ** 2
print(z.requires_grad)   # and this?

True
False


## 7. detach vs no_grad

In [20]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 3

y_detached = y.detach()

In [21]:
y.requires_grad

True

In [22]:
y_detached.requires_grad

False

In [23]:
y.backward()

# Exercise Set 4 — Mini Training Loop

## 8. Train a straight line

In [26]:
import torch

# Data — 10 points on y = 3x + 1
x_train = torch.linspace(0, 1, 10).unsqueeze(1)  # shape (10, 1)
y_train = 3 * x_train + 1

# Model — one linear layer (learns weight w and bias b)
model = torch.nn.Linear(1, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
loss_fn = torch.nn.MSELoss()

# Training loop — fill in the 5 steps
for epoch in range(200):
    optimizer.zero_grad() # 1. zero gradients
    pred = model(x_train) # 2. forward pass
    loss=loss_fn(pred,y_train) # 3. compute loss
    loss.backward() # 4. backward pass
    optimizer.step() # 5. optimizer step

    if epoch % 40 == 0:
        print(f"Epoch {epoch} | Loss: {loss.item():.4f}")

# After training, what did the model learn?
print(f"\nLearned weight: {model.weight.item():.4f}")   # should be ≈ 3.0
print(f"Learned bias:   {model.bias.item():.4f}")       # should be ≈ 1.0

Epoch 0 | Loss: 8.0730
Epoch 40 | Loss: 0.1181
Epoch 80 | Loss: 0.0325
Epoch 120 | Loss: 0.0089
Epoch 160 | Loss: 0.0025

Learned weight: 2.9194
Learned bias:   1.0438
